#Práctica Módulo IA: Fine-tuning de un modelo de lenguaje Gemma 2B


In [81]:
!pip install -qU unsloth datasets trl accelerate bitsandbytes scikit-learn

In [100]:
import torch
import pandas as pd

from datasets import load_dataset
from sklearn.metrics import accuracy_score

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

In [83]:
dataset = load_dataset(
    "fancyzhx/ag_news"
)

print(dataset)

print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [86]:
labels = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

In [85]:
raw_train_dataset = dataset["train"]

raw_test_dataset = dataset["test"]

In [87]:
def format_dataset(example):

    return {

        "text": f"""### Instruction:
Classify the news category.

### News:
{example['text']}

### Response:
{labels[example['label']]}"""

    }

In [88]:
train_dataset = raw_train_dataset.map(
    format_dataset
)


test_dataset = raw_test_dataset.map(
    format_dataset
)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [90]:
# Reducir tamaño para T4
train_dataset = train_dataset.select(
    range(5000)
)


test_dataset = test_dataset.select(
    range(500)
)

In [91]:
# Cargar Gemma 2B
MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"


model, tokenizer = FastLanguageModel.from_pretrained(

    model_name=MODEL_NAME,

    max_seq_length=512,

    load_in_4bit=True

)

==((====))==  Unsloth 2026.7.5: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


In [92]:
# Función de pregunta (zero-shot)
def preguntar_modelo(texto):


    prompt = f"""
You are a news classifier.

Choose ONLY one category:

World
Sports
Business
Sci/Tech


News:
{texto}


Category:
"""


    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")


    outputs = model.generate(

        **inputs,

        max_new_tokens=5,

        do_sample=False,

        pad_token_id=tokenizer.eos_token_id

    )

    respuesta = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True

    )

    return respuesta

In [93]:
def limpiar_respuesta(resp):

    resp = resp.lower()


    if "sports" in resp:
        return "Sports"


    if "world" in resp:
        return "World"


    if "business" in resp:
        return "Business"


    if "tech" in resp or "sci" in resp:
        return "Sci/Tech"


    return "Unknown"

In [94]:
def evaluar(dataset):

    predicciones = []

    reales = []


    for x in dataset:


        respuesta = preguntar_modelo(
            x["text"]
        )


        predicciones.append(
            limpiar_respuesta(respuesta)
        )


        reales.append(
            labels[x["label"]]
        )


    return (

        accuracy_score(
            reales,
            predicciones
        ),

        predicciones,

        reales

    )

In [95]:
FastLanguageModel.for_inference(model)

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_atte

In [96]:
# Accuracy antes del fine tuning
accuracy_before, pred_before, real_before = evaluar(
    raw_test_dataset.select(range(200))
)


print(
    "Accuracy antes:",
    accuracy_before
)

Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documen

Accuracy antes: 0.265


In [97]:
# Aplicamos QLoRA
model = FastLanguageModel.get_peft_model(

    model,

    r=16,


    target_modules=[

        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"

    ],


    lora_alpha=16,

    lora_dropout=0,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=42

)

In [98]:
# Vemos los parametros
model.print_trainable_parameters()

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


In [99]:
# Entrenamiento

trainer = SFTTrainer(

    model=model,

    tokenizer=tokenizer,

    train_dataset=train_dataset,

    dataset_text_field="text",

    max_seq_length=512,


    args=TrainingArguments(

        per_device_train_batch_size=2,

        gradient_accumulation_steps=4,

        learning_rate=2e-4,

        warmup_steps=10,

        num_train_epochs=1,

        logging_steps=10,

        fp16=True,

        output_dir="gemma_agnews",

        optim="adamw_8bit"

    )

)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [101]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
10,28.385577
20,24.137639
30,17.445297
40,12.298829
50,9.463108
60,8.183133
70,7.891746
80,7.526232
90,7.364275
100,7.098082


PicklingError: Can't pickle <class 'trl.trainer.sft_config.SFTConfig'>: it's not the same object as trl.trainer.sft_config.SFTConfig

In [102]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [103]:
# Accuracy después
accuracy_after, pred_after, real = evaluar(
    test_dataset
)


print(
    "Accuracy después:",
    accuracy_after
)

Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=5) a

Accuracy después: 0.29


In [104]:
# Comparación final
resultado = pd.DataFrame({

    "Modelo":[
        "Gemma base",
        "Gemma + QLoRA"
    ],

    "Accuracy":[
        accuracy_before,
        accuracy_after
    ]

})


resultado

,Modelo,Accuracy
0,Gemma base,0.265
1,Gemma + QLoRA,0.290


In [105]:
# Ejemplo:
ejemplos = pd.DataFrame({

    "Noticia": test_dataset["text"][:10],

    "Real":[
        labels[x]
        for x in test_dataset["label"][:10]
    ],

    "Antes":pred_before[:10],

    "Después":pred_after[:10]

})


ejemplos

,Noticia,Real,Antes,Después
0,### Instruction:\nClassify the news category.\...,Business,Sports,Sports
1,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
2,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
3,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
4,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
5,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
6,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
7,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
8,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports
9,### Instruction:\nClassify the news category.\...,Sci/Tech,Sports,Sports


In [106]:
for i in range(10):

    texto = test_dataset[i]["text"]

    print("====================")
    print("NOTICIA:")
    print(texto[:200])

    print("\nREAL:")
    print(labels[test_dataset[i]["label"]])

    print("\nMODELO:")
    print(preguntar_modelo(texto))

Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NOTICIA:
### Instruction:
Classify the news category.

### News:
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent fi

REAL:
Business

MODELO:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a news classifier.

Choose ONLY one category:

World
Sports
Business
Sci/Tech


News:
### Instruction:
Classify the news category.

### News:
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.

### Response:
Business


Category:






NOTICIA:
### Instruction:
Classify the news category.

### News:
The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeer

REAL:
Sci/Tech

MODELO:


Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


You are a news classifier.

Choose ONLY one category:

World
Sports
Business
Sci/Tech


News:
### Instruction:
Classify the news category.

### News:
Loosing the War on Terrorism \\"Sven Jaschan, self-confessed author of the Netsky and Sasser viruses, is\responsible for 70 percent of virus infections in 2004, according to a six-month\virus roundup published Wednesday by antivirus company Sophos."\\"The 18-year-old Jaschan was taken into custody in Germany in May by police who\said he had admitted programming both the Netsky and Sasser worms, something\experts at Microsoft confirmed. (A Microsoft antivirus reward program led to the\teenager's arrest.) During the five months preceding Jaschan's capture, there\were at least 25 variants of Netsky and one of the port-scanning network worm\Sasser."\\"Graham Cluley, senior technology consultant at Sophos, said it was staggeri ...\\

### Response:
Sci/Tech


Category:






NOTICIA:
### Instruction:
Classify the news category.

### News:
FOAF

Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a news classifier.

Choose ONLY one category:

World
Sports
Business
Sci/Tech


News:
### Instruction:
Classify the news category.

### News:
FOAFKey: FOAF, PGP, Key Distribution, and Bloom Filters \\FOAF/LOAF  and bloom filters have a lot of interesting properties for social\network and whitelist distribution.\\I think we can go one level higher though and include GPG/OpenPGP key\fingerpring distribution in the FOAF file for simple web-of-trust based key\distribution.\\What if we used FOAF and included the PGP key fingerprint(s) for identities?\This could mean a lot.  You include the PGP key fingerprints within the FOAF\file of your direct friends and then include a bloom filter of the PGP key\fingerprints of your entire whitelist (the source FOAF file would of course need\to be encrypted ).\\Your whitelist would be populated from the social network as your client\discovered new identit ...\\

### Response:
Sci/Tech


Category:






NOTICIA:
### Instruction:
Classify the new

Both `max_new_tokens` (=5) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a news classifier.

Choose ONLY one category:

World
Sports
Business
Sci/Tech


News:
### Instruction:
Classify the news category.

### News:
E-mail scam targets police chief Wiltshire Police warns about "phishing" after its fraud squad chief was targeted.

### Response:
Sci/Tech


Category:






NOTICIA:
### Instruction:
Classify the news category.

### News:
Card fraud unit nets 36,000 cards In its first two years, the UK's dedicated card fraud unit, has recovered 36,000 stolen cards and 171 arrests 

REAL:
Sci/Tech

MODELO:

You are a news classifier.

Choose ONLY one category:

World
Sports
Business
Sci/Tech


News:
### Instruction:
Classify the news category.

### News:
Card fraud unit nets 36,000 cards In its first two years, the UK's dedicated card fraud unit, has recovered 36,000 stolen cards and 171 arrests - and estimates it saved 65m.

### Response:
Sci/Tech


Category:








En esta práctica se ha realizado un fine-tuning de un modelo de lenguaje Gemma 2B mediante la técnica QLoRA utilizando Unsloth para adaptarlo a una tarea específica de clasificación de noticias.

Se ha utilizado el dataset AG News de Hugging Face, compuesto por noticias clasificadas en cuatro categorías: World, Sports, Business y Sci/Tech. Primero se evaluó el modelo base sin entrenamiento adicional y posteriormente se realizó un ajuste fino mediante capas LoRA.

Finalmente, se compararon los resultados utilizando la métrica accuracy, observando la mejora del modelo después del fine-tuning:

Gemma base: 26,5% de accuracy
Gemma + QLoRA: 29% de accuracy

El experimento demuestra cómo un modelo generalista puede especializarse en una tarea concreta mediante técnicas eficientes de ajuste de parámetros sin necesidad de entrenar todos sus pesos.